In [0]:
# Define the Silver checkpoint location
silver_checkpoint = "abfss://medallion-data@boodystorage.dfs.core.windows.net/checkpoints/silver"

# 1. Read incrementally from the Bronze table
bronze_stream = spark.readStream.table("nyc_taxi_lakehouse.bronze.taxi_trips_raw")

# 2. Cleanse and transform the data
silver_cleansed = (
    bronze_stream
    # Remove records with missing or invalid business metrics
    .filter("passenger_count > 0")
    .filter("trip_distance > 0.0")
    .filter("fare_amount >= 0.0")
    # Drop rows where critical timestamp fields are null
    .dropna(subset=["tpep_pickup_datetime", "tpep_dropoff_datetime"])
)

# 3. Write the cleansed stream to the Silver Delta table
query = (
    silver_cleansed.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", f"{silver_checkpoint}/data")
    .trigger(availableNow=True)
    .toTable("nyc_taxi_lakehouse.silver.taxi_trips_cleansed")
)

query.awaitTermination()
print("Silver cleansing and ingestion complete!")

Silver cleansing and ingestion complete!


In [0]:
%sql
SELECT 
    MIN(passenger_count) AS min_passengers,
    MIN(trip_distance) AS min_distance,
    MIN(fare_amount) AS min_fare,
    COUNT(*) AS total_clean_records
FROM nyc_taxi_lakehouse.silver.taxi_trips_cleansed;

min_passengers,min_distance,min_fare,total_clean_records
1,0.01,0.0,2724215
